# SMILES HypeCheck: CATE Estimation and Uplift Targeting

This notebook checks whether CATE models that estimate individual effects well are also the best models for campaign targeting decisions.

The analysis has two parts:

1. **Synthetic sanity check** - true individual treatment effects are known, so PEHE and targeting regret can be measured directly.
2. **Hillstrom randomized campaign pilot** - true CATE is unobserved, so the evaluation uses randomized uplift metrics: Qini AUC, AUUC, and Uplift@20.

The main question is not only *which model has the best score*, but whether the preferred model changes when the decision metric changes.


## 1. Environment and Reproducibility

The notebook uses CausalPFN, scikit-uplift, scikit-learn, PyTorch, NumPy, pandas, and matplotlib. The installation cell is intentionally kept separate so it can be skipped when the environment is already prepared.


In [ ]:
%pip install -q causalpfn scikit-uplift


In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from causalpfn import CATEEstimator
from sklift.datasets import fetch_hillstrom
from sklift.metrics import (
    qini_auc_score,
    qini_curve,
    uplift_at_k,
    uplift_auc_score,
    uplift_curve,
)
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    train_test_split,
)

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
SYNTHETIC_N = 5_000
SYNTHETIC_D = 5
SYNTHETIC_TEST_SIZE = 0.30

TARGETING_FRACTION = 0.20

HILLSTROM_SAMPLE_SIZE = 12_000
HILLSTROM_FOLDS = 2

CAUSALPFN_CONTEXT_SIZE = 512
CAUSALPFN_QUERY_SIZE = 256
CAUSALPFN_NEIGHBOURS = 128

BOOTSTRAP_ITERATIONS = 1_000
BOOTSTRAP_SEED = 42

MODEL_ORDER = [
    'CausalPFN',
    'S-Learner',
    'T-Learner',
]

COLORS = {
    'CausalPFN': '#ffbf00',
    'S-Learner': '#5b8ff9',
    'T-Learner': '#61d9a5',
}


## 2. Synthetic Sanity Check

This section uses simulated data with known true CATE. That makes PEHE meaningful and allows a direct comparison between estimation quality and targeting quality.

The synthetic experiment is a pipeline check, not the main empirical claim. It answers: can the notebook train models, produce CATE predictions, and evaluate them against known ground truth?


In [ ]:
X = rng.normal(
    0,
    1,
    size=(SYNTHETIC_N, SYNTHETIC_D),
).astype(np.float32)

tau = (
    np.sin(X[:, 0])
    + 0.5 * X[:, 1]
    - 0.25 * X[:, 2] ** 2
).astype(np.float32)

treatment = rng.binomial(
    1,
    0.5,
    size=SYNTHETIC_N,
).astype(np.float32)

y0 = (
    X[:, 0]
    - X[:, 1]
    + 0.5 * X[:, 2]
    + rng.normal(0, 0.1, size=SYNTHETIC_N)
).astype(np.float32)

y = (y0 + treatment * tau).astype(np.float32)

(
    X_train,
    X_test,
    t_train,
    t_test,
    y_train,
    y_test,
    tau_train,
    tau_test,
) = train_test_split(
    X,
    treatment,
    y,
    tau,
    test_size=SYNTHETIC_TEST_SIZE,
    random_state=SEED,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print(f'Mean true CATE on test: {tau_test.mean():.4f}')


### 2.1 CausalPFN on Synthetic Data

CausalPFN is fit on the training split and evaluated on the holdout split. If the returned object is a PyTorch tensor, it is moved to CPU and converted to a one-dimensional NumPy array.


In [ ]:
causalpfn_synthetic = CATEEstimator(
    device=device,
    verbose=True,
)

causalpfn_synthetic.fit(
    X_train,
    t_train,
    y_train,
)

cate_causalpfn = causalpfn_synthetic.estimate_cate(X_test)

if torch.is_tensor(cate_causalpfn):
    cate_causalpfn = cate_causalpfn.detach().cpu().numpy()

cate_causalpfn = np.asarray(cate_causalpfn).reshape(-1)

print('Predictions:', len(cate_causalpfn))


### 2.2 Synthetic Baselines

Two classical baselines are used for comparison:

- **S-Learner**: one outcome model with treatment as an additional feature.
- **T-Learner**: separate outcome models for treated and control units.


In [ ]:
def create_synthetic_regressor():
    return HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=SEED,
    )


s_learner_synthetic = create_synthetic_regressor()
s_learner_synthetic.fit(
    np.column_stack([X_train, t_train]),
    y_train,
)

X_test_treated = np.column_stack([
    X_test,
    np.ones(len(X_test), dtype=np.float32),
])
X_test_control = np.column_stack([
    X_test,
    np.zeros(len(X_test), dtype=np.float32),
])

cate_s_learner = (
    s_learner_synthetic.predict(X_test_treated)
    - s_learner_synthetic.predict(X_test_control)
)

t_learner_control = create_synthetic_regressor()
t_learner_treated = create_synthetic_regressor()

control_mask = t_train == 0
treated_mask = t_train == 1

t_learner_control.fit(
    X_train[control_mask],
    y_train[control_mask],
)
t_learner_treated.fit(
    X_train[treated_mask],
    y_train[treated_mask],
)

cate_t_learner = (
    t_learner_treated.predict(X_test)
    - t_learner_control.predict(X_test)
)


In [ ]:
def evaluate_synthetic_model(name, prediction, true_tau, fraction=TARGETING_FRACTION):
    prediction = np.asarray(prediction).reshape(-1)
    k = max(1, int(len(true_tau) * fraction))

    oracle_top = np.argsort(true_tau)[-k:]
    model_top = np.argsort(prediction)[-k:]

    oracle_value = true_tau[oracle_top].mean()
    model_value = true_tau[model_top].mean()
    regret = oracle_value - model_value

    return {
        'Model': name,
        'PEHE': np.sqrt(np.mean((prediction - true_tau) ** 2)),
        'Value@20': model_value,
        'Oracle Value@20': oracle_value,
        'Regret@20': regret,
        'Relative regret, %': regret / abs(oracle_value) * 100,
    }


synthetic_results_df = pd.DataFrame([
    evaluate_synthetic_model('CausalPFN', cate_causalpfn, tau_test),
    evaluate_synthetic_model('S-Learner', cate_s_learner, tau_test),
    evaluate_synthetic_model('T-Learner', cate_t_learner, tau_test),
])

synthetic_results_df['PEHE rank'] = (
    synthetic_results_df['PEHE']
    .rank(method='min', ascending=True)
    .astype(int)
)
synthetic_results_df['Targeting rank'] = (
    synthetic_results_df['Value@20']
    .rank(method='min', ascending=False)
    .astype(int)
)

synthetic_results_df = synthetic_results_df.sort_values('PEHE rank')
display(synthetic_results_df.round(4))

best_pehe_model = synthetic_results_df.loc[
    synthetic_results_df['PEHE'].idxmin(),
    'Model',
]
best_targeting_model = synthetic_results_df.loc[
    synthetic_results_df['Value@20'].idxmax(),
    'Model',
]

print('Best by PEHE:', best_pehe_model)
print('Best by Value@20:', best_targeting_model)
print('Different leaders:', best_pehe_model != best_targeting_model)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    synthetic_results_df['Model'],
    synthetic_results_df['PEHE'],
    color=[COLORS[model] for model in synthetic_results_df['Model']],
)
axes[0].set_title('Synthetic CATE estimation accuracy')
axes[0].set_ylabel('PEHE, lower is better')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(
    synthetic_results_df['Model'],
    synthetic_results_df['Value@20'],
    color=[COLORS[model] for model in synthetic_results_df['Model']],
)
axes[1].axhline(
    synthetic_results_df['Oracle Value@20'].iloc[0],
    color='black',
    linestyle='--',
    label='Oracle',
)
axes[1].set_title('Synthetic top-20% targeting value')
axes[1].set_ylabel('True CATE in selected group')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

plt.tight_layout()
plt.show()


### 2.3 Synthetic Interpretation

In the current pilot run, CausalPFN is the best synthetic model both by PEHE and by top-20% targeting value. This validates the synthetic evaluation pipeline, but it does **not** support the hypothesis that the best CATE estimator and the best targeter must differ.

The real-data section below is the more relevant test for marketing uplift targeting, because it evaluates the models on a randomized campaign benchmark.


## 3. Hillstrom Randomized Campaign Pilot

The Hillstrom dataset is a randomized email marketing experiment. This notebook uses the **Women's E-Mail vs No E-Mail** contrast and the binary **visit** outcome.

Important difference from the synthetic experiment: individual true CATE is not observed. Therefore PEHE cannot be computed on Hillstrom. The evaluation uses randomized uplift metrics instead.


In [ ]:
X_raw, y_raw, treatment_raw = fetch_hillstrom(
    target_col='visit',
    return_X_y_t=True,
)

selected_groups = [
    'Womens E-Mail',
    'No E-Mail',
]

hillstrom_mask = treatment_raw.isin(selected_groups)

X_hillstrom = X_raw.loc[hillstrom_mask].reset_index(drop=True)
y_hillstrom = (
    y_raw.loc[hillstrom_mask]
    .reset_index(drop=True)
    .astype(np.int32)
)
t_hillstrom = (
    treatment_raw.loc[hillstrom_mask]
    .reset_index(drop=True)
    .eq('Womens E-Mail')
    .astype(np.int32)
)

X_hillstrom = pd.get_dummies(
    X_hillstrom,
    drop_first=False,
).astype(np.float32)

group_summary = (
    pd.DataFrame({
        'treatment': t_hillstrom,
        'visit': y_hillstrom,
    })
    .groupby('treatment')
    .agg(['mean', 'count'])
)

print('Dataset shape:', X_hillstrom.shape)
print(f'Treatment share: {t_hillstrom.mean():.4f}')
print(f'Visit rate: {y_hillstrom.mean():.4f}')
display(group_summary)


### 3.1 Honest Out-of-Fold Design

The real-data pilot uses out-of-fold predictions. Each row receives a CATE or uplift score from a model that did not train on that row.

The pilot is intentionally constrained:

- stratified 12k subsample;
- two folds;
- one random seed;
- CausalPFN context budget of 512.

These constraints keep the experiment feasible, but they limit statistical reliability and generalizability.


In [ ]:
X_full = X_hillstrom.to_numpy(dtype=np.float32)
y_full = y_hillstrom.to_numpy(dtype=np.float32)
t_full = t_hillstrom.to_numpy(dtype=np.float32)

full_strata = (
    t_full.astype(np.int32) * 2
    + y_full.astype(np.int32)
)

sampler = StratifiedShuffleSplit(
    n_splits=1,
    train_size=HILLSTROM_SAMPLE_SIZE,
    random_state=SEED,
)

sample_idx, _ = next(sampler.split(X_full, full_strata))

X_data = X_full[sample_idx]
y_data = y_full[sample_idx]
t_data = t_full[sample_idx]
strata = full_strata[sample_idx]

print('Pilot sample:', X_data.shape)
print(f'Treatment share: {t_data.mean():.4f}')
print(f'Visit rate: {y_data.mean():.4f}')


In [ ]:
def create_classifier():
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=SEED,
    )


def fit_predict_causalpfn(X_train_fold, t_train_fold, y_train_fold, X_test_fold):
    model = CATEEstimator(
        device=device,
        verbose=True,
        max_context_length=CAUSALPFN_CONTEXT_SIZE,
        max_query_length=CAUSALPFN_QUERY_SIZE,
        num_neighbours=CAUSALPFN_NEIGHBOURS,
    )
    model.fit(
        X=X_train_fold,
        t=t_train_fold,
        y=y_train_fold,
    )
    prediction = model.estimate_cate(X=X_test_fold)

    if torch.is_tensor(prediction):
        prediction = prediction.detach().cpu().numpy()

    return np.asarray(prediction).reshape(-1)


def fit_predict_s_learner(X_train_fold, t_train_fold, y_train_fold, X_test_fold):
    model = create_classifier()
    model.fit(
        np.column_stack([X_train_fold, t_train_fold]),
        y_train_fold,
    )

    X_test_treated = np.column_stack([
        X_test_fold,
        np.ones(len(X_test_fold), dtype=np.float32),
    ])
    X_test_control = np.column_stack([
        X_test_fold,
        np.zeros(len(X_test_fold), dtype=np.float32),
    ])

    return (
        model.predict_proba(X_test_treated)[:, 1]
        - model.predict_proba(X_test_control)[:, 1]
    )


def fit_predict_t_learner(X_train_fold, t_train_fold, y_train_fold, X_test_fold):
    control_model = create_classifier()
    treated_model = create_classifier()

    control_mask = t_train_fold == 0
    treated_mask = t_train_fold == 1

    control_model.fit(
        X_train_fold[control_mask],
        y_train_fold[control_mask],
    )
    treated_model.fit(
        X_train_fold[treated_mask],
        y_train_fold[treated_mask],
    )

    return (
        treated_model.predict_proba(X_test_fold)[:, 1]
        - control_model.predict_proba(X_test_fold)[:, 1]
    )


In [ ]:
oof_predictions = {
    'CausalPFN': np.zeros(len(X_data), dtype=np.float32),
    'S-Learner': np.zeros(len(X_data), dtype=np.float32),
    'T-Learner': np.zeros(len(X_data), dtype=np.float32),
}

cv = StratifiedKFold(
    n_splits=HILLSTROM_FOLDS,
    shuffle=True,
    random_state=SEED,
)

for fold, (train_idx, test_idx) in enumerate(
    tqdm(cv.split(X_data, strata), total=HILLSTROM_FOLDS),
    start=1,
):
    print(f'Fold {fold}/{HILLSTROM_FOLDS}')

    X_train_fold = X_data[train_idx]
    X_test_fold = X_data[test_idx]
    y_train_fold = y_data[train_idx]
    t_train_fold = t_data[train_idx]

    oof_predictions['CausalPFN'][test_idx] = fit_predict_causalpfn(
        X_train_fold,
        t_train_fold,
        y_train_fold,
        X_test_fold,
    )
    oof_predictions['S-Learner'][test_idx] = fit_predict_s_learner(
        X_train_fold,
        t_train_fold,
        y_train_fold,
        X_test_fold,
    )
    oof_predictions['T-Learner'][test_idx] = fit_predict_t_learner(
        X_train_fold,
        t_train_fold,
        y_train_fold,
        X_test_fold,
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 4. Randomized Uplift Metrics

Hillstrom has randomized treatment assignment, so uplift metrics can compare targeting policies without observing true individual CATE.

Metrics used here:

- **Qini AUC**: overall ranking quality for incremental outcomes.
- **AUUC**: area under the uplift curve.
- **Uplift@20**: estimated incremental visit-rate lift when targeting the top 20% by model score.


In [ ]:
def calculate_uplift_metrics(y_true, treatment, prediction):
    return {
        'Qini AUC': qini_auc_score(
            y_true=y_true,
            uplift=prediction,
            treatment=treatment,
        ),
        'AUUC': uplift_auc_score(
            y_true=y_true,
            uplift=prediction,
            treatment=treatment,
        ),
        'Uplift@20, pp': uplift_at_k(
            y_true=y_true,
            uplift=prediction,
            treatment=treatment,
            strategy='overall',
            k=TARGETING_FRACTION,
        ) * 100,
    }


point_metrics = {
    model_name: calculate_uplift_metrics(
        y_data,
        t_data,
        prediction,
    )
    for model_name, prediction in oof_predictions.items()
}

metrics_df = pd.DataFrame([
    {'Model': model_name, **metrics}
    for model_name, metrics in point_metrics.items()
])

metrics_df['Qini rank'] = (
    metrics_df['Qini AUC']
    .rank(ascending=False, method='min')
    .astype(int)
)
metrics_df['Uplift@20 rank'] = (
    metrics_df['Uplift@20, pp']
    .rank(ascending=False, method='min')
    .astype(int)
)

metrics_df = metrics_df.sort_values('Qini rank').reset_index(drop=True)
display(metrics_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

qini_random_end = None

for model_name in MODEL_ORDER:
    prediction = oof_predictions[model_name]

    qini_x, qini_y = qini_curve(
        y_true=y_data,
        uplift=prediction,
        treatment=t_data,
    )
    axes[0].plot(
        qini_x / len(y_data),
        qini_y,
        label=model_name,
        linewidth=2,
        color=COLORS[model_name],
    )
    if qini_random_end is None:
        qini_random_end = qini_y[-1]

    uplift_x, uplift_y = uplift_curve(
        y_true=y_data,
        uplift=prediction,
        treatment=t_data,
    )
    axes[1].plot(
        uplift_x / len(y_data),
        uplift_y,
        label=model_name,
        linewidth=2,
        color=COLORS[model_name],
    )

axes[0].plot(
    [0, 1],
    [0, qini_random_end],
    linestyle='--',
    color='black',
    label='Random targeting',
)

average_uplift = y_data[t_data == 1].mean() - y_data[t_data == 0].mean()
axes[1].axhline(
    average_uplift,
    linestyle='--',
    color='black',
    label='Average treatment effect',
)

axes[0].set_title('Hillstrom: Qini curves')
axes[0].set_xlabel('Targeted population fraction')
axes[0].set_ylabel('Cumulative incremental visits')
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].set_title('Hillstrom: uplift curves')
axes[1].set_xlabel('Targeted population fraction')
axes[1].set_ylabel('Estimated uplift')
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


## 5. Bootstrap Confidence Intervals

The point estimates are not enough to claim that one model is statistically better. This section uses a stratified paired bootstrap over the already generated out-of-fold predictions.

The bootstrap preserves treatment/control balance by resampling treated and control rows separately, then recomputes all metrics on the same sampled rows for every model.

Limitation: this bootstrap captures sampling uncertainty conditional on the fitted out-of-fold predictions. It does not fully capture model-training variability from additional seeds, folds, or subsamples.


In [ ]:
bootstrap_rng = np.random.default_rng(BOOTSTRAP_SEED)

control_indices = np.where(t_data == 0)[0]
treated_indices = np.where(t_data == 1)[0]

metric_names = [
    'Qini AUC',
    'AUUC',
    'Uplift@20, pp',
]

bootstrap_values = {
    model_name: {
        metric: np.zeros(BOOTSTRAP_ITERATIONS, dtype=np.float64)
        for metric in metric_names
    }
    for model_name in MODEL_ORDER
}

for bootstrap_index in tqdm(
    range(BOOTSTRAP_ITERATIONS),
    desc='Bootstrap',
):
    sampled_control = bootstrap_rng.choice(
        control_indices,
        size=len(control_indices),
        replace=True,
    )
    sampled_treated = bootstrap_rng.choice(
        treated_indices,
        size=len(treated_indices),
        replace=True,
    )
    sampled_indices = np.concatenate([
        sampled_control,
        sampled_treated,
    ])
    bootstrap_rng.shuffle(sampled_indices)

    y_bootstrap = y_data[sampled_indices]
    t_bootstrap = t_data[sampled_indices]

    for model_name in MODEL_ORDER:
        prediction_bootstrap = oof_predictions[model_name][sampled_indices]
        current_metrics = calculate_uplift_metrics(
            y_bootstrap,
            t_bootstrap,
            prediction_bootstrap,
        )

        for metric in metric_names:
            bootstrap_values[model_name][metric][bootstrap_index] = current_metrics[metric]


In [ ]:
summary_rows = []

for model_name in MODEL_ORDER:
    row = {'Model': model_name}

    for metric in metric_names:
        values = bootstrap_values[model_name][metric]
        row[metric] = point_metrics[model_name][metric]
        row[f'{metric} CI low'] = np.percentile(values, 2.5)
        row[f'{metric} CI high'] = np.percentile(values, 97.5)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df['Qini rank'] = (
    summary_df['Qini AUC']
    .rank(ascending=False, method='min')
    .astype(int)
)
summary_df = summary_df.sort_values('Qini rank').reset_index(drop=True)

formatted_summary = pd.DataFrame({
    'Model': summary_df['Model'],
    'Qini AUC [95% CI]': summary_df.apply(
        lambda row: (
            f'{row["Qini AUC"]:.4f} '
            f'[{row["Qini AUC CI low"]:.4f}; '
            f'{row["Qini AUC CI high"]:.4f}]'
        ),
        axis=1,
    ),
    'AUUC [95% CI]': summary_df.apply(
        lambda row: (
            f'{row["AUUC"]:.4f} '
            f'[{row["AUUC CI low"]:.4f}; '
            f'{row["AUUC CI high"]:.4f}]'
        ),
        axis=1,
    ),
    'Uplift@20, pp [95% CI]': summary_df.apply(
        lambda row: (
            f'{row["Uplift@20, pp"]:.2f} '
            f'[{row["Uplift@20, pp CI low"]:.2f}; '
            f'{row["Uplift@20, pp CI high"]:.2f}]'
        ),
        axis=1,
    ),
    'Qini rank': summary_df['Qini rank'],
})

display(formatted_summary)


In [ ]:
comparison_specs = [
    {
        'Metric': 'Qini AUC',
        'First': 'S-Learner',
        'Second': 'CausalPFN',
        'Direction': 'S-Learner minus CausalPFN',
    },
    {
        'Metric': 'AUUC',
        'First': 'S-Learner',
        'Second': 'CausalPFN',
        'Direction': 'S-Learner minus CausalPFN',
    },
    {
        'Metric': 'Uplift@20, pp',
        'First': 'CausalPFN',
        'Second': 'S-Learner',
        'Direction': 'CausalPFN minus S-Learner',
    },
]

comparison_rows = []

for spec in comparison_specs:
    metric = spec['Metric']
    first = spec['First']
    second = spec['Second']

    point_difference = point_metrics[first][metric] - point_metrics[second][metric]
    bootstrap_difference = (
        bootstrap_values[first][metric]
        - bootstrap_values[second][metric]
    )

    ci_low = np.percentile(bootstrap_difference, 2.5)
    ci_high = np.percentile(bootstrap_difference, 97.5)

    if ci_low > 0:
        interpretation = 'First model is better in this paired bootstrap'
    elif ci_high < 0:
        interpretation = 'Second model is better in this paired bootstrap'
    else:
        interpretation = 'No statistically clear difference'

    comparison_rows.append({
        'Comparison': spec['Direction'],
        'Metric': metric,
        'Difference': point_difference,
        '95% CI low': ci_low,
        '95% CI high': ci_high,
        'Interpretation': interpretation,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path.cwd() / "artifacts" / "local" / "model_comparison" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

random_qini_end = None

for model_name in MODEL_ORDER:
    prediction = oof_predictions[model_name]
    qini_x, qini_y = qini_curve(
        y_true=y_data,
        uplift=prediction,
        treatment=t_data,
    )
    axes[0].plot(
        qini_x / len(y_data),
        qini_y,
        label=model_name,
        color=COLORS[model_name],
        linewidth=2,
    )

    if random_qini_end is None:
        random_qini_end = qini_y[-1]

axes[0].plot(
    [0, 1],
    [0, random_qini_end],
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='Random targeting',
)
axes[0].set_title('Hillstrom: out-of-fold Qini curves')
axes[0].set_xlabel('Targeted population fraction')
axes[0].set_ylabel('Cumulative incremental visits')
axes[0].legend()
axes[0].grid(alpha=0.2)

plot_summary = summary_df.set_index('Model').loc[MODEL_ORDER]

uplift_values = plot_summary['Uplift@20, pp'].to_numpy()
uplift_ci_low = plot_summary['Uplift@20, pp CI low'].to_numpy()
uplift_ci_high = plot_summary['Uplift@20, pp CI high'].to_numpy()

lower_errors = uplift_values - uplift_ci_low
upper_errors = uplift_ci_high - uplift_values
x_positions = np.arange(len(MODEL_ORDER))

axes[1].bar(
    x_positions,
    uplift_values,
    color=[COLORS[model] for model in MODEL_ORDER],
    alpha=0.9,
)
axes[1].errorbar(
    x_positions,
    uplift_values,
    yerr=np.vstack([lower_errors, upper_errors]),
    fmt='none',
    color='black',
    capsize=5,
    linewidth=1.5,
)

average_uplift_pp = (
    y_data[t_data == 1].mean()
    - y_data[t_data == 0].mean()
) * 100
axes[1].axhline(
    average_uplift_pp,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label=f'Overall ATE: {average_uplift_pp:.2f} pp',
)
axes[1].set_xticks(x_positions, MODEL_ORDER, rotation=15)
axes[1].set_title('Hillstrom: uplift at 20% budget')
axes[1].set_ylabel('Uplift, percentage points')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'hillstrom_pilot_results.png',
    dpi=300,
    bbox_inches='tight',
)
plt.show()


In [ ]:
summary_df.to_csv(
    OUTPUT_DIR / 'hillstrom_pilot_metrics_ci.csv',
    index=False,
)
comparison_df.to_csv(
    OUTPUT_DIR / 'hillstrom_pilot_comparisons.csv',
    index=False,
)

configuration_df = pd.DataFrame([{
    'Dataset': 'Hillstrom',
    'Treatment': 'Womens E-Mail',
    'Control': 'No E-Mail',
    'Outcome': 'visit',
    'Pilot sample size': len(y_data),
    'Evaluation': '2-fold honest splitting',
    'Random seed': SEED,
    'CausalPFN context size': CAUSALPFN_CONTEXT_SIZE,
    'CausalPFN query size': CAUSALPFN_QUERY_SIZE,
    'CausalPFN neighbours': CAUSALPFN_NEIGHBOURS,
    'Bootstrap iterations': BOOTSTRAP_ITERATIONS,
}])
configuration_df.to_csv(
    OUTPUT_DIR / 'hillstrom_pilot_configuration.csv',
    index=False,
)

print('Saved files in', OUTPUT_DIR)
print('hillstrom_pilot_results.png')
print('hillstrom_pilot_metrics_ci.csv')
print('hillstrom_pilot_comparisons.csv')
print('hillstrom_pilot_configuration.csv')


## 6. How Reliable Are These Results?

| Level | Assessment | Why |
|---|---|---|
| Technical correctness | Good | Hillstrom is a real randomized campaign benchmark, treatment assignment is balanced, and the real-data evaluation uses out-of-fold predictions. |
| Statistical reliability | Improved but still limited | Bootstrap CIs are included, but the experiment still uses one stratified 12k subsample, two folds, and one seed. The bootstrap is conditional on the fitted OOF predictions and does not fully measure training instability. |
| Generalizability | Low | The pilot covers only Hillstrom, one treatment contrast, and one binary outcome. |
| Main hypothesis check | Partial | The synthetic setting has true CATE but the same model leads by PEHE and targeting. The real RCT has no true CATE, so PEHE cannot be compared with Qini directly. |

### Current Hillstrom Pilot Reading

In the current run, S-Learner has the highest point estimates for overall Qini and AUUC. CausalPFN has the highest point estimate for Uplift@20.

The key Uplift@20 gap is small:

```text
5.4835 - 5.2500 = 0.2335 percentage points
```

The paired bootstrap interval for **CausalPFN minus S-Learner** at Uplift@20 crosses zero. Therefore, this pilot does not show that CausalPFN is statistically better than S-Learner at a 20% targeting budget.


## 7. Honest Claims

### What can be claimed

- The evaluation pipeline works on a real randomized benchmark.
- On this 12k Hillstrom pilot, S-Learner has the best point estimate for Qini AUC and AUUC.
- On this 12k Hillstrom pilot, CausalPFN has the best point estimate for Uplift@20.
- Model choice may depend on the targeting budget and selected decision metric.
- The result motivates a broader robustness check.

### What cannot be claimed yet

- CausalPFN is statistically better than S-Learner at a 20% targeting budget.
- S-Learner is universally better for uplift modeling.
- The hypothesis `best estimator != best targeter` is confirmed.
- The result transfers to other campaigns, outcomes, or treatment contrasts.


## 8. Pre-Defense Summary

**Title:** Preliminary validation on a real randomized campaign

**Main text:** On a 12k Hillstrom pilot, S-Learner achieved the highest overall Qini and AUUC, while CausalPFN produced the highest point estimate of uplift at a 20% targeting budget. The Uplift@20 difference between CausalPFN and S-Learner was small, about +0.23 percentage points, and was not statistically clear under the paired bootstrap interval.

**Conclusion:** The pilot validates the evaluation pipeline and indicates that model selection may depend on the targeting budget and decision metric.

**Footnote:** Hillstrom Women's Email vs No Email, visit outcome, stratified 12k subsample, 2-fold honest splitting, one seed, CausalPFN context budget 512.

**Oral explanation:** We are not claiming that the main hypothesis is confirmed yet. The first real-data experiment shows that the infrastructure works on randomized campaign data and that different decision metrics can point to different models. The next stage is to test whether this pattern is stable using more seeds, more folds or repeated subsamples, additional baselines, and other datasets.


## 9. Recommended Next Steps

1. Repeat the Hillstrom pilot across multiple random seeds and repeated stratified subsamples.
2. Increase the number of folds if runtime allows.
3. Add more uplift baselines, for example causal forests, X-Learner, DR-Learner, and uplift random forests.
4. Evaluate more targeting budgets, for example 5%, 10%, 20%, and 30%.
5. Add at least one more randomized campaign dataset to test external validity.
6. Report training-time and inference-time costs, because targeting performance may not be the only operational criterion.
